# Example of a Rice cooker DT #

**Description**

**Characteristics**


**Steps for the example to execute**
1) Export all libraries

In [1]:
#Import our modules
import src.Service as Service
import src.GlOb as Glob

# # import exeAreas
import src.exeMgn as exeMgn
import src.Comm as Comm

# #import modules for API
from src.API import srcAPI as src
from src.API import sinkAPI as sink
from src.API import FMUAPI
from src.API import MatlabAPI
from src.API import PythonAPI
# #import necessary services such as Server to connect remote components
from src.API.comm_protos.TCP import TCPServer as Server

#import general modules from python
import time
import os

2) Create the interfaces for all the components (this is a manual step) 

    a) Import libraries for models created in python 

In [2]:
from BreweryMod import BiProducts, kalman_filter

b) Create the adaptors for each of the components

In [ ]:
#manually set by user
dir_brewery = os.getcwd() + '\\' + 'BreweryMod' 

############## sensor inteface ##################################################


############### Sink interface ##################################################

####################Models in Python ############################################
biProduct_obj = BiProducts.biProducts()
# encapsulate the model to store attributes:
inputNames =['c_sugar']
outputNames = ['conc_yeast','conc_etOH', 'ABV_etOH']
biProduct_model = PythonAPI.Wrapping(model=biProduct_obj, directory=dir_brewery,
                                      modelName='biProductCal',inputsNames=inputNames, 
                                      outputsNames=outputNames, exeSchedule={1:'execute'},
                                      printFlag=True)
biProduct_name = 'biProductCal'
biProduct_model.save_to_csv(file_name=biProduct_name)


kalman_obj = kalman_filter.KalmanFilter()
inputNames = ["x_measure", "x_cal"]
outputNames = ["x_hat"]

kalman_model = PythonAPI.Wrapping(model=kalman_obj, directory=dir_brewery,
                                      modelName='kalmanFilter',inputsNames=inputNames, 
                                      outputsNames=outputNames, exeSchedule={1:'step'},
                                        printFlag=True)
kalman_name = 'kalmanFilter'
kalman_model.save_to_csv(file_name=kalman_name)

#################### Models in FMU ##############################################
control = FMUAPI.FMU(modelName= 'control_modelFMU', directory=dir_brewery)
thermoDyn = FMUAPI.FMU(modelName= 'FMUThermoDynamics', directory=dir_brewery)
sugarFerm = FMUAPI.FMU(modelName= 'suggarFerm', directory=dir_brewery)



Data of the class has been extracted
Inputs and outputs have been defined for this model
{'ports': ['c_sugar', 'conc_etOH', 'conc_yeast', 'ABV_etOH', 'param'], 'methods': ['execute', 'yeast_initial']}
Data of the class has been extracted
Inputs and outputs have been defined for this model
{'ports': ['x_measure', 'x_cal', 'x_hat', 'param'], 'methods': ['reset', 'step']}


In [ ]:
#extract FMU info (input, output and parameter names, types, units)
control.get_info(infoType="Variables")
# thermoDyn.get_info(infoType="Variables")
# sugarFerm.get_info(infoType="Variables")


*The first step is to create the component manager*

In [16]:
#Component Manager
directory = os.getcwd() + '\\' + 'BreweryMod' 

#instansiate all the components: by defining inputs, outputs and parameters 
# always as a dictionary {'name': , 'unit': , 'datatype'}

####################### #sensor component #########################################


################################ sink component #####################################


######################## Data process component ####################################


######################### Python model: biproduct###########################
input1 = {'name':"c_sugar", 'unit':"mol/L", 'datatype':"float",'val':'1011.83'}

inputs = [input1]

output1 = {'name':"conc_etOH", 'unit':"mol/L", 'datatype':"float",'val':''}
output2 ={'name':"conc_yeast", 'unit':"mol/L", 'datatype':"float",'val':''}
output3 = {'name':"ABV_etOH", 'unit':"-", 'datatype':"float",'val':''}
# outputs = [output1,output2]
outputs = [output1,output2,output3]

parameter1 = {'name':"sug_init", 'unit':"mol/L", 'datatype':"float",'val':'1011.832'}
parameter2 = {'name':"yeast_grams", 'unit':"g", 'datatype':"float",'val':'11'}
parameter3 = {'name':"volume_squaremeters", 'unit':"m^2", 'datatype':"float",'val':'0.025'}

parameters = [parameter1,parameter2,parameter3]

biProduct = Comm.Model(name = 'biProductCal',SimE= "Python",modelDir = directory, 
                    inputs=inputs,outputs=outputs, parameters= parameters)

######################### Python model: kalman filter###########################
input1 = {'name':"x_measure", 'unit':"mol/L", 'datatype':"float",'val':'1011.83'}
input2 = {'name':"x_cal", 'unit':"mol/L", 'datatype':"float",'val':'1011.83'}
inputs = [input1,input2]


output1 ={'name':"x_hat", 'unit':"mol/L", 'datatype':"float",'val':''}
outputs = [output1]

parameter1 = {'name':"P_prev", 'unit':"-", 'datatype':"float",'val':'1648.12'}
parameter2 = {'name':"sigma_sensor_default", 'unit':"-", 'datatype':"float",'val':'34.2'}

parameters = [parameter1,parameter2]

kalmanFilter = Comm.Model(name = 'kalmanFilter',SimE= "Python",modelDir = directory, 
                    inputs=inputs,outputs=outputs, parameters= parameters)

########################FMU Control model#############################################
# [ModelVariable(name='T_measure', type='Real'),
#  ModelVariable(name='T_control', type='Real'),
#  ModelVariable(name='freq', type='Real'),
#  ModelVariable(name='u', type='Real'),
input1 = {'name':"T_measure", 'unit':"K", 'datatype':"float",'val':''}
input2 = {'name':"T_control", 'unit':"K", 'datatype':"float",'val':''}
input3 = {'name':"freq", 'unit':"min", 'datatype':"int",'val':''}
inputs = [input1,input2,input3]

output1 = {'name':"u", 'unit':"-", 'datatype':"int",'val':''}
outputs = [output1]


parameters = []

controlModel = Comm.Model(name = 'control_modelFMU', SimE = 'FMU', modelDir=directory,
                    inputs=inputs, outputs=outputs, parameters=parameters)

########################  FMU ThermoDynamics #############################################
# [ModelVariable(name='T_k', type='Real'),
#  ModelVariable(name='delta_C', type='Real'),
#  ModelVariable(name='pump', type='Real'),
#  ModelVariable(name='T_cool', type='Real'),
#  ModelVariable(name='T_amb', type='Real'),
#  ModelVariable(name='density', type='Real'),
#  ModelVariable(name='T_k1', type='Real'),
input1 = {'name':"T_k", 'unit':"K", 'datatype':"float",'val':''}
input2 = {'name':"delta_C", 'unit':"mol/Lmin", 'datatype':"float",'val':''}
input3 = {'name':"pump", 'unit':"-", 'datatype':"float",'val':''}
input4 = {'name':"T_cool", 'unit':"K", 'datatype':"float",'val':''}
input5 = {'name':"T_amb", 'unit':"K", 'datatype':"float",'val':''}
input6 = {'name':"density", 'unit':"kg/m^3", 'datatype':"float",'val':''}
inputs = [input1,input2]

output1 = {'name':"T_k1", 'unit':"K", 'datatype':"float",'val':''}
outputs = [output1]

# ModelVariable(name='A', type='Real'),
#  ModelVariable(name='A_amb', type='Real'),
#  ModelVariable(name='Cp', type='Real'),
#  ModelVariable(name='Vol', type='Real'),
#  ModelVariable(name='delta_H', type='Real'),
#  ModelVariable(name='h', type='Real'),
#  ModelVariable(name='h_amb', type='Real'),

parameter1 = {'name':"A", 'unit':"m^2", 'datatype':"float",'val':'0.31'}
parameter2 = {'name':"A_amb", 'unit':"m^2", 'datatype':"float",'val':'0.754'}
parameter3 = {'name':"Cp", 'unit':"kJ/kg K", 'datatype':"float",'val':'3.8'}
parameter4 = {'name':"Vol", 'unit':"m^3", 'datatype':"float",'val':'0.025'}
parameter5 = {'name':"delta_H", 'unit':"kJ/mol", 'datatype':"float",'val':'-65'}
parameter6 = {'name':"h", 'unit':"W/m^2 K", 'datatype':"float",'val':'678.07'}
parameter7 = {'name':"h_amb", 'unit':"W/m^2 K", 'datatype':"float",'val':'42.93'}
parameters = [parameter1,parameter2,parameter3,parameter4,parameter5,parameter6,parameter7]

thermoDynamics = Comm.Model(name = 'FMUThermoDynamics', SimE = 'FMU', modelDir=directory,
                    inputs=inputs, outputs=outputs, parameters=parameters)


######################## FMu sugar Fermentation  #############################################
# [ModelVariable(name='temperature', type='Real'),
#  ModelVariable(name='c_conc', type='Real'),
#  ModelVariable(name='delta_c_conc', type='Real'),
#  ModelVariable(name='time', type='Real'),
#  ModelVariable(name='sug_current', type='Real'),
#  ModelVariable(name='t_step', type='Real')]

input1 = {'name':"T", 'unit':"K", 'datatype':"float",'val':''}
inputs = [input1]

output1 = {'name':"c_conc", 'unit':"mol/L", 'datatype':"float",'val':''}
output2 = {'name':"delta_c_conc", 'unit':"mol/L min", 'datatype':"float",'val':''}
outputs = [output1,output2]

parameter1 = {'name':"sug_current", 'unit':"mol/L", 'datatype':"float",'val':'1011.83'}

parameters = [parameter1]

sugarFermentation = Comm.Model(name = 'suggarFerm', SimE = 'FMU', modelDir=directory,
                    inputs=inputs, outputs=outputs, parameters=parameters)



Data of the class has been extracted
Inputs and outputs have been defined for this model
Data of the class has been extracted
Inputs and outputs have been defined for this model
Model control_modelFMU has no parameters defined, in case is wrong correct it


*Communication manager*

*Execution manager*